# 02 — Push Casino Events via Zerobus Ingest SDK

Generates the **same synthetic casino events** as Part 1 (same seed, same
fields) but instead of writing files and streaming them in, we **push them
directly** to the Lakehouse using the Zerobus Python SDK (gRPC).

No Kafka. No file drops. No intermediate infrastructure.
Just: create table → push data.

All data is **synthetic** and does not represent any real organization.

In [1]:
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
from dotenv import load_dotenv
from faker import Faker
import os
import random
import uuid
import time
from datetime import datetime, timedelta

load_dotenv(dotenv_path="../.env")

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")
ZEROBUS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_zerobus"

SERVER_ENDPOINT = os.getenv("ZEROBUS_SERVER_ENDPOINT")
WORKSPACE_URL   = os.getenv("DATABRICKS_HOST")
CLIENT_ID       = os.getenv("ZEROBUS_CLIENT_ID")
CLIENT_SECRET   = os.getenv("ZEROBUS_CLIENT_SECRET")

## Same synthetic data generator as Part 1

Using the same seed so the data is comparable.

In [2]:
fake = Faker()
Faker.seed(42)
random.seed(42)

CASINO_FLOORS = ["Main Floor", "High Roller", "VIP Lounge", "Penny Lane", "Sports Zone"]
GAME_TYPES = ["Slots", "Video Poker", "Electronic Roulette", "Electronic Blackjack", "Keno"]
MACHINE_IDS = [f"MCH-{i:04d}" for i in range(1, 51)]
PLAYER_IDS = [f"PLR-{fake.unique.random_int(min=10000, max=99999)}" for _ in range(100)]


def generate_event(base_time):
    game_type = random.choice(GAME_TYPES)
    bet_ranges = {
        "Slots": (0.25, 50.00),
        "Video Poker": (1.00, 100.00),
        "Electronic Roulette": (5.00, 500.00),
        "Electronic Blackjack": (10.00, 1000.00),
        "Keno": (1.00, 20.00),
    }
    min_bet, max_bet = bet_ranges[game_type]
    bet_amount = round(random.uniform(min_bet, max_bet), 2)

    outcome_roll = random.random()
    if outcome_roll < 0.55:
        win_amount = 0.0
    elif outcome_roll < 0.80:
        win_amount = round(bet_amount * random.uniform(0.5, 2.0), 2)
    elif outcome_roll < 0.95:
        win_amount = round(bet_amount * random.uniform(2.0, 5.0), 2)
    else:
        win_amount = round(bet_amount * random.uniform(5.0, 20.0), 2)

    event_time = base_time + timedelta(seconds=random.randint(0, 59))

    return {
        "event_id": str(uuid.uuid4()),
        "machine_id": random.choice(MACHINE_IDS),
        "casino_floor": random.choice(CASINO_FLOORS),
        "game_type": game_type,
        "player_id": random.choice(PLAYER_IDS),
        "bet_amount": bet_amount,
        "win_amount": win_amount,
        "event_timestamp": event_time.isoformat(),
    }

## Connect to Zerobus and open a stream

The SDK handles OAuth authentication and gRPC connection automatically.
We just provide the endpoint, workspace URL, and service principal credentials.

In [3]:
sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)

table_props = TableProperties(ZEROBUS_TABLE)
options = StreamConfigurationOptions(record_type=RecordType.JSON)

stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_props, options)
print(f"Zerobus stream opened to {ZEROBUS_TABLE}")

2026-03-27T16:43:30.699444Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1
Zerobus stream opened to alexander_booth.streaming_demo.slot_events_zerobus
2026-03-27T16:43:30.699804Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1
2026-03-27T16:43:30.699901Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1


## Push 500 events directly to the Lakehouse

Each `ingest_record_offset()` call pushes a record and returns an offset
confirming it was durably received. No files, no message bus — just push.

In [4]:
NUM_EVENTS = 500
base_time = datetime(2026, 3, 25, 10, 0, 0)

start = time.time()

try:
    for i in range(NUM_EVENTS):
        batch_time = base_time + timedelta(minutes=i // 50)
        event = generate_event(batch_time)
        offset = stream.ingest_record_offset(event)

        if (i + 1) % 100 == 0:
            print(f"  Pushed {i + 1}/{NUM_EVENTS} events (latest offset: {offset})")

    stream.flush()
    elapsed = time.time() - start
    print(f"\nDone! Pushed {NUM_EVENTS} events in {elapsed:.1f} seconds")
    print(f"Throughput: {NUM_EVENTS / elapsed:.0f} events/sec")

finally:
    stream.close()
    print("Stream closed.")

  Pushed 100/500 events (latest offset: 99)
  Pushed 200/500 events (latest offset: 199)
  Pushed 300/500 events (latest offset: 299)
  Pushed 400/500 events (latest offset: 399)
  Pushed 500/500 events (latest offset: 499)
2026-03-27T16:43:30.730272Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1
2026-03-27T16:43:30.901002Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 169. Waiting for offset 499. stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1

Done! Pushed 500 events in 0.4 seconds2026-03-27T16:43:31.100401Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Flush completed. stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1

Throughput: 1299 events/sec
2026-03-27T16:43:31.100627Z  INFO databricks_zerobus_ingest_sdk: Closing stream stream_id=85e5241f-0df0-4109-8fe2-c0c58c8fa0c1
2026-03-27T16:43:31.100632Z  INFO databricks_zerobus_in

That's it. No Kafka broker, no file staging, no intermediate infrastructure.
Just a gRPC stream from wherever your data lives to a Delta table in the Lakehouse.